In [ ]:
# BLOCK 0 – Dependencies
# pip install opencv-python nibabel numpy pandas torch totalsegmentator ultralytics tqdm matplotlib

In [ ]:
# BLOCK 1 – Setup imports, paths, flags, and constants

import gc, io, time, traceback, contextlib
import traceback
from pathlib import Path
import cv2
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from totalsegmentator.python_api import totalsegmentator
from ultralytics import YOLO
from tqdm import tqdm
import shutil, time
import matplotlib.pyplot as plt


# ── PATHS (EDIT HERE) ────────────────────────────────────────
MODEL_PATH = Path("PATH_TO_MODEL_WEIGHTS.pt")
NIFTI_DIR  = Path("PATH_TO_NIFTI_DIRECTORY")
CSV_PATH = NIFTI_DIR / "overscanning_results.csv"

# ── FLAGS ─────────────────────────────────────────────────────
DISPLAY_DETECTION = True
FAST_MODEL = False
MULTI_LABEL_MASK = True

# ── CONSTANTS ─────────────────────────────────────────────────
FINAL_CONF = 0.20
BACKGROUND_HU = -300
model = YOLO(str(MODEL_PATH))

In [39]:
# BLOCK 2 – Pubic symphysis detection + femur fallback → caudal overscan

def run_ts_silent(*args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return totalsegmentator(*args, **kwargs)

def preprocess_slice(arr: np.ndarray) -> np.ndarray:
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    arr = (arr * 255).astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)

def is_ct_vol(p: Path) -> bool:
    if p.parent.name.startswith("ts_"):
        return False
    if p.name.endswith("_combined.nii.gz"):
        return False
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True

def ensure_femur_mask(ct_path: Path) -> Path | None:
    out_dir     = ct_path.parent / "ts_femur"
    fem_l_path  = out_dir / "femur_left.nii.gz"
    fem_r_path  = out_dir / "femur_right.nii.gz"
    merged_path = ct_path.parent / "femur_combined.nii.gz"

    if merged_path.exists():
        return merged_path

    if not (fem_l_path.exists() and fem_r_path.exists()):
        out_dir.mkdir(exist_ok=True)
        for dev in ("gpu", "cpu"):
            try:
                run_ts_silent(
                    ct_path, out_dir,
                    roi_subset=["femur_left", "femur_right"],
                    task="total",
                    fast=FAST_MODEL,
                    device=dev,
                )
                break
            except Exception as e:
                print(f"TS({dev}) {ct_path.name}: {e}")
        else:
            return None

    try:
        fem_l = nib.load(fem_l_path).get_fdata() > 0
        fem_r = nib.load(fem_r_path).get_fdata() > 0
    except FileNotFoundError:
        return None

    merged = (fem_l | fem_r).astype(np.uint8)
    if not merged.any():
        return None

    ref = nib.load(fem_l_path if fem_l_path.exists() else fem_r_path)
    nib.save(nib.Nifti1Image(merged, ref.affine, ref.header), merged_path)

    for p in (fem_l_path, fem_r_path):
        if p.exists():
            p.unlink()
    if out_dir.exists() and not any(out_dir.iterdir()):
        out_dir.rmdir()
    return merged_path

def femur_top_info(ct_path: Path) -> tuple[int, float] | None:
    m = ensure_femur_mask(ct_path)
    if m is None:
        return None
    mask = nib.load(str(m))
    mask_np = mask.get_fdata() > 0
    slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if slices.size == 0:
        return None
    affine = mask.affine
    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in slices]
    return max(z_coords, key=lambda t: t[1])

DEVICE = 0 if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu"

def find_valid_pubic_slice(ct_path: Path, z_cutoff_mm: float) -> int | None:
    ct = nib.load(str(ct_path))
    affine = ct.affine
    vol = ct.get_fdata()
    H, W, Z = vol.shape

    best_conf, best_slice = -1.0, None
    for z in range(Z):
        if float((affine @ [0, 0, z, 1])[2]) > z_cutoff_mm:
            continue
        img = preprocess_slice(vol[:, :, z])
        res = model.predict(img, conf=FINAL_CONF, device=DEVICE, save=False, verbose=False)[0]
        for b in sorted(res.boxes, key=lambda bb: float(bb.conf), reverse=True):
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            if vol[int((y1+y2)/2), int((x1+x2)/2), z] <= BACKGROUND_HU:
                continue
            cx = int((x1+x2)/2)
            if abs(cx - W // 2) > 0.20 * W:
                continue
            win = vol[max(0, int((y1+y2)/2) - 10):min(H, int((y1+y2)/2) + 10),
                      max(0, cx - 10):min(W, cx + 10), z]
            if win.mean() < 150:
                continue
            conf = float(b.conf)
            if conf > best_conf:
                best_conf, best_slice = conf, z
            break
    return best_slice

def process_single_case(ct_path: Path) -> dict | None:
    try:
        fem_data = femur_top_info(ct_path)
        if fem_data:
            fem_slice, fem_top_z = fem_data
            z_cut = fem_top_z
        else:
            fem_slice, fem_top_z = None, np.nan
            z_cut = float("inf")

        pubic_slice = find_valid_pubic_slice(ct_path, z_cut)
        if pubic_slice is None and fem_slice is not None:
            pubic_slice, source_label = fem_slice, "FemurFallback"
        elif pubic_slice is None:
            return None
        else:
            source_label = "YOLO" if not np.isnan(fem_top_z) else "YOLO_NoFemur"

        ct_img  = nib.load(str(ct_path))
        affine  = ct_img.affine
        Z       = ct_img.shape[2]
        pubic_z = float((affine @ [0, 0, pubic_slice, 1])[2])
        end_z   = min(float((affine @ [0, 0, k, 1])[2]) for k in range(Z))
        caudal  = abs(end_z - pubic_z)

        return {
            "file_name": ct_path.name,
            "pubic_z_mm": int(round(pubic_z)),
            "scan_end_z_mm": int(round(end_z)),
            "caudal_overscan_mm": int(round(caudal)),
            "femur_top_z_mm": (int(round(fem_top_z))
                               if not np.isnan(fem_top_z) else np.nan),
            "pubic_source": source_label,
        }

    except Exception:
        traceback.print_exc(limit=1)
        return None
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_batch():
    patterns = ("*.nii.gz", "*.nii")
    ct_files = sorted({
        p for pat in patterns
        for p in NIFTI_DIR.rglob(pat)
        if p.is_file() and is_ct_vol(p)
    })

    if not ct_files:
        print(f"No matching NIfTI files in {NIFTI_DIR}")
        return

    caudal_cols = ["pubic_z_mm", "scan_end_z_mm", "caudal_overscan_mm",
                   "femur_top_z_mm", "pubic_source"]

    if CSV_PATH.exists():
        df_prev = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
        for c in caudal_cols:
            if c not in df_prev.columns:
                df_prev[c] = np.nan
    else:
        df_prev = pd.DataFrame(columns=["file_name"] + caudal_cols)

    done_mask = (~df_prev[caudal_cols].isna()).all(axis=1)
    done_set  = set(df_prev.loc[done_mask, "file_name"])

    results = []
    t0 = time.time()

    for ct_path in tqdm(ct_files, desc="Processing caudal overscan", unit="vol"):
        if ct_path.name in done_set:
            continue
        row = process_single_case(ct_path)
        if row:
            results.append(row)

    if not results:
        print("No new successful cases.")
        return

    df_new = pd.DataFrame(results)
    df_out = df_prev.merge(df_new, on="file_name", how="outer", suffixes=("", "_new"))

    for col in caudal_cols:
        new_col = col + "_new"
        if new_col in df_out.columns:
            mask = df_out[col].isna()
            df_out.loc[mask, col] = df_out.loc[mask, new_col]
            df_out.drop(columns=[new_col], inplace=True)

    df_out.sort_values("file_name").to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"Finished. CSV now contains {len(df_out)} rows")
    print(f"Total time: {time.time() - t0:.1f}s")

if __name__ == "__main__":
    run_batch()

Processing caudal overscan: 100%|██████████| 2/2 [00:00<?, ?vol/s]

No new successful cases.


In [37]:
# BLOCK 3 – Liver & spleen segmentation → cranial overscan

def run_ts_silent(*args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return totalsegmentator(*args, **kwargs)

def nifti_basename(p: Path) -> str:
    n = p.name
    low = n.lower()
    if low.endswith(".nii.gz"):
        return n[:-7]
    if low.endswith(".nii"):
        return n[:-4]
    return p.stem

def file_matches_parent(p: Path) -> bool:
    return nifti_basename(p).lower() == p.parent.name.lower()

def is_ct_vol(p: Path) -> bool:
    if p.parent.name.startswith("ts_"):
        return False
    if p.name.startswith("ts_"):
        return False
    if p.name.endswith("_combined.nii.gz"):
        return False
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True

def ensure_liver_spleen_mask(ct_path: Path) -> Path | None:
    out_dir      = ct_path.parent / "ts_liver_spleen"
    liver_mask   = out_dir / "liver.nii.gz"
    spleen_mask  = out_dir / "spleen.nii.gz"
    merged_path  = ct_path.parent / "liver_spleen_combined.nii.gz"

    if merged_path.exists():
        return merged_path

    if not (liver_mask.exists() and spleen_mask.exists()):
        out_dir.mkdir(exist_ok=True)
        for dev in ("gpu", "cpu"):
            try:
                run_ts_silent(
                    ct_path,
                    out_dir,
                    roi_subset=["liver", "spleen"],
                    task="total",
                    fast=FAST_MODEL,
                    device=dev,
                )
                break
            except Exception as e:
                print(f"TS({dev}) {ct_path.name}: {e}")
        else:
            return None

    try:
        liver_data  = nib.load(liver_mask ).get_fdata() > 0
        spleen_data = nib.load(spleen_mask).get_fdata() > 0
    except FileNotFoundError:
        return None

    if MULTI_LABEL_MASK:
        combined = np.zeros(liver_data.shape, np.uint8)
        combined[liver_data]  = 1
        combined[spleen_data] = 2
    else:
        combined = (liver_data | spleen_data).astype(np.uint8)

    ref_img = nib.load(liver_mask if liver_mask.exists() else spleen_mask)
    nib.save(nib.Nifti1Image(combined, ref_img.affine, ref_img.header), merged_path)

    for p in (liver_mask, spleen_mask):
        if p.exists():
            p.unlink()
    if out_dir.exists() and not any(out_dir.iterdir()):
        out_dir.rmdir()

    return merged_path

def cranial_overscan(ct_path: Path, mask_path: Path) -> tuple[int, int, int, str]:
    ct_img   = nib.load(str(ct_path))
    mask_img = nib.load(str(mask_path))
    affine   = ct_img.affine
    mask_np  = mask_img.get_fdata()

    seg_slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if seg_slices.size == 0:
        raise RuntimeError("combined mask empty")

    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in seg_slices]

    Z = ct_img.shape[2]
    z_edge0 = float((affine @ [0, 0,      0, 1])[2])
    z_edgeN = float((affine @ [0, 0, Z - 1, 1])[2])
    cranial_edge_z = max(z_edge0, z_edgeN)

    highest_slice, highest_z = min(z_coords, key=lambda t: abs(t[1] - cranial_edge_z))

    labels     = mask_np[:, :, highest_slice][mask_np[:, :, highest_slice] > 0].astype(int)
    organ_map  = {1: "Liver", 2: "Spleen"}
    organ_top  = organ_map.get(int(np.bincount(labels).argmax()), "Unknown")

    cranial_mm    = int(round(abs(cranial_edge_z - highest_z)))
    scan_start_mm = int(round(cranial_edge_z))
    organ_z_mm    = int(round(highest_z))
    return cranial_mm, organ_z_mm, scan_start_mm, organ_top

def process_single_case(ct_path: Path) -> dict | None:
    try:
        mask_path = ct_path.parent / "liver_spleen_combined.nii.gz"
        if not mask_path.exists():
            mask_path = ensure_liver_spleen_mask(ct_path)
            if mask_path is None or not mask_path.exists():
                return None

        cranial_mm, organ_z_mm, scan_start_mm, organ_top = cranial_overscan(ct_path, mask_path)

        return {
            "file_name"          : ct_path.name,
            "liver_spleen_z_mm"  : organ_z_mm,
            "scan_start_z_mm"    : scan_start_mm,
            "cranial_overscan_mm": cranial_mm,
            "top_organ"          : organ_top,
        }

    except Exception:
        traceback.print_exc(limit=1)
        return None
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_batch():
    patterns = ("*.nii.gz", "*.nii")
    ct_files = sorted({
        p for pat in patterns
        for p in NIFTI_DIR.rglob(pat)
        if p.is_file() and file_matches_parent(p) and is_ct_vol(p)
    })

    if not ct_files:
        print(f"No matching NIfTI files in {NIFTI_DIR}")
        return

    cranial_cols = ["liver_spleen_z_mm", "scan_start_z_mm", "cranial_overscan_mm", "top_organ"]

    if CSV_PATH.exists():
        df_prev = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
        for c in cranial_cols:
            if c not in df_prev.columns:
                df_prev[c] = np.nan
    else:
        df_prev = pd.DataFrame(columns=["file_name"] + cranial_cols)

    done_mask = (~df_prev[cranial_cols].isna()).all(axis=1)
    done_set  = set(df_prev.loc[done_mask, "file_name"])

    results = []
    t0 = time.time()

    for ct_path in tqdm(ct_files, desc="Processing cranial overscan", unit="vol"):
        if ct_path.name in done_set:
            continue
        row = process_single_case(ct_path)
        if row:
            results.append(row)

    if not results:
        print("No new successful cases.")
        return

    df_new = pd.DataFrame(results)
    df_out = df_prev.merge(df_new, on="file_name", how="outer", suffixes=("", "_new"))

    if "top_organ" in df_out.columns:
        df_out["top_organ"] = df_out["top_organ"].astype(object)

    for col in cranial_cols:
        new_col = col + "_new"
        if new_col in df_out.columns:
            if df_out[col].dtype != df_out[new_col].dtype:
                df_out[col] = df_out[col].astype(object)
            mask = df_out[col].isna()
            df_out.loc[mask, col] = df_out.loc[mask, new_col]
            df_out.drop(columns=[new_col], inplace=True)

    df_out.sort_values("file_name").to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"Finished. CSV now contains {len(df_out)} rows")
    print(f"Total time: {time.time() - t0:.1f}s")

if __name__ == "__main__":
    run_batch()

Processing cranial overscan: 100%|██████████| 2/2 [00:00<?, ?vol/s]

No new successful cases.


In [38]:
# BLOCK 5 – Generate preview MP4s of mid-coronal slices (ABDOMEN)

OUT_DIR = NIFTI_DIR.parent / "trauma_overscan_videos_test"
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)

print(f"{len(df)} CT volumes found")


def build_mp4_abdomen(scan_id: str,
                      pubic_z_mm: float | None,
                      organ_z_mm: float | None,
                      organ_label: str | None,
                      pubic_source: str | None,
                      fps: int = 48,
                      slice_span: int = 100):
    folder = NIFTI_DIR / scan_id
    if not folder.is_dir():
        raise FileNotFoundError(f"{folder} not found")

    candidates = [f for f in folder.glob("*.nii*")
                  if f.name.startswith(scan_id)
                  and "_combined" not in f.name.lower()
                  and not f.name.startswith("ts_")]
    if not candidates:
        raise FileNotFoundError("CT volume not found")
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple CT volumes: {[c.name for c in candidates]}")

    ct_path = candidates[0]

    fem_path = None
    try: fem_path = ensure_femur_mask(ct_path)
    except Exception: pass
    org_path = None
    try: org_path = ensure_liver_spleen_mask(ct_path)
    except Exception: pass

    mp4_path = OUT_DIR / f"{scan_id}.mp4"
    if mp4_path.exists():
        mp4_path.unlink()

    ct_img  = nib.load(str(ct_path))
    vol     = ct_img.get_fdata()
    fem_msk = nib.load(str(fem_path)).get_fdata() > 0 if fem_path and fem_path.exists() else None
    org_msk = nib.load(str(org_path)).get_fdata() > 0 if org_path and org_path.exists() else None
    vx, _, vz = ct_img.header.get_zooms()[:3]

    _, Y, Z = vol.shape
    z_world = np.flip((ct_img.affine @ np.vstack([
        np.zeros(Z), np.zeros(Z), np.arange(Z), np.ones(Z)
    ]))[2])

    pubic_row = int(np.argmin(np.abs(z_world - pubic_z_mm))) if (pubic_z_mm is not None and np.isfinite(pubic_z_mm)) else None
    organ_row = int(np.argmin(np.abs(z_world - organ_z_mm))) if (organ_z_mm is not None and np.isfinite(organ_z_mm)) else None

    mid_y, half = Y // 2, slice_span // 2
    start_y, end_y = max(0, mid_y - half), min(Y - 1, mid_y + half)
    y_stretch = vz / vx

    font, fs, th = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
    red, green = (0, 0, 255), (0, 255, 0)
    femur_color = (255, 0, 0)
    organ_color = (0, 255, 255)
    ALPHA = 0.35
    landmark_name = "Femur" if (pubic_source == "FemurFallback") else "Pubic Symphysis"

    def render(y_idx: int):
        base = np.clip((np.flipud(vol[:, y_idx, :].T) + 200) / 500, 0, 1) * 255
        base = cv2.cvtColor(base.astype(np.uint8), cv2.COLOR_GRAY2BGR)

        overlay = np.zeros_like(base, dtype=np.uint8)
        if fem_msk is not None:
            mask_fem = np.flipud(fem_msk[:, y_idx, :].T)
            overlay[mask_fem] = femur_color
        if org_msk is not None:
            mask_org = np.flipud(org_msk[:, y_idx, :].T)
            overlay[mask_org] = organ_color

        img = cv2.addWeighted(base, 1.0, overlay, ALPHA, 0)

        if y_stretch != 1.0:
            h, w = img.shape[:2]
            img = cv2.resize(img, (w, int(h * y_stretch)), interpolation=cv2.INTER_CUBIC)

        h, w = img.shape[:2]
        if pubic_row is not None:
            y_line = int(pubic_row * y_stretch)
            cv2.line(img, (0, y_line), (w - 1, y_line), red, 2)
            cv2.putText(img, f"{landmark_name} z={pubic_z_mm:.0f} mm",
                        (10, max(20, y_line - 6)), font, fs, red, th, cv2.LINE_AA)

        if organ_row is not None and organ_label:
            y_line = int(organ_row * y_stretch)
            cv2.line(img, (0, y_line), (w - 1, y_line), green, 2)
            cv2.putText(img, f"{organ_label} z={organ_z_mm:.0f} mm",
                        (10, min(h - 10, y_line + 20)), font, fs, green, th, cv2.LINE_AA)

        cv2.putText(img, f"{scan_id} | y={y_idx}",
                    (10, h - 10), font, fs, (255, 255, 0), th, cv2.LINE_AA)
        return img

    first = render(start_y)
    h, w = first.shape[:2]
    vw = cv2.VideoWriter(str(mp4_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    vw.write(first)
    for y in range(start_y + 1, end_y + 1):
        vw.write(render(y))
    vw.release()

ok = failed = 0
for _, r in tqdm(df.iterrows(), total=len(df), desc="Processing (abdomen MP4s)", unit="scan"):
    sid = r["file_name"].split(".nii")[0]
    try:
        build_mp4_abdomen(
            sid,
            float(r["pubic_z_mm"])        if "pubic_z_mm"        in r and pd.notna(r["pubic_z_mm"])        else None,
            float(r["liver_spleen_z_mm"]) if "liver_spleen_z_mm" in r and pd.notna(r["liver_spleen_z_mm"]) else None,
            str(r["top_organ"]).strip()   if "top_organ"         in r and pd.notna(r["top_organ"])         else None,
            str(r["pubic_source"]).strip()if "pubic_source"      in r and pd.notna(r["pubic_source"])      else None,
        )
        ok += 1
    except Exception as e:
        failed += 1
        tqdm.write(f"✗ {sid}: {e}")
        traceback.print_exc()

print(f"Finished. {ok} MP4s created and saved to {OUT_DIR}")

2 CT volumes found


Processing (abdomen MP4s): 100%|██████████| 2/2 [00:01<00:00,  1.50scan/s]

Finished. 2 MP4s created and saved to D:\Ryan\trauma_overscan_videos_test


In [31]:
# BLOCK 6 – Overscan metrics & summary stats

SUMMARY_CSV_PATH = CSV_PATH.with_name("summary_statistics.csv")

df = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)

needed_cols = [
    "pubic_source", "caudal_overscan_mm", "cranial_overscan_mm",
    "scan_end_z_mm", "scan_start_z_mm"
]
for c in needed_cols:
    if c not in df.columns:
        df[c] = pd.NA

caudal_thresh = (
    df["pubic_source"].eq("FemurFallback").map({True: 50, False: 30})
    if "pubic_source" in df.columns else pd.Series(30, index=df.index)
)

df["caudal_overscan?"]  = (pd.to_numeric(df["caudal_overscan_mm"], errors="coerce")  > caudal_thresh).map({True: "yes", False: "no"})
df["cranial_overscan?"] =  pd.to_numeric(df["cranial_overscan_mm"],  errors="coerce").gt(30).map({True: "yes", False: "no"})
df["overscanning?"]     = ((df["caudal_overscan?"] == "yes") | (df["cranial_overscan?"] == "yes")).map({True: "yes", False: "no"})

df["calc_caudal_overscan_mm"]  = (pd.to_numeric(df["caudal_overscan_mm"],  errors="coerce") - caudal_thresh).clip(lower=0)
df["calc_cranial_overscan_mm"] = (pd.to_numeric(df["cranial_overscan_mm"], errors="coerce") - 30).clip(lower=0)
df["calc_total_overscan_mm"]   = df["calc_caudal_overscan_mm"] + df["calc_cranial_overscan_mm"]

scan_length = (pd.to_numeric(df["scan_end_z_mm"], errors="coerce") -
               pd.to_numeric(df["scan_start_z_mm"], errors="coerce")).replace(0, pd.NA)

percent_vals = (df["calc_total_overscan_mm"] / scan_length * 100).abs().round()
df["%_overscan"] = percent_vals.apply(lambda x: f"{int(x)}%" if pd.notna(x) else pd.NA)

with open(CSV_PATH, encoding="utf-8-sig") as fh:
    first_line = fh.readline()
sep = "\t" if "\t" in first_line else ","
df.to_csv(CSV_PATH, index=False, sep=sep, encoding="utf-8-sig")
print("Main CSV updated.")

caudal_excess  = df["calc_caudal_overscan_mm"]
cranial_excess = df["calc_cranial_overscan_mm"]
total_excess   = df["calc_total_overscan_mm"]

m_caudal = caudal_excess [caudal_excess  > 0].mean()
s_caudal = caudal_excess [caudal_excess  > 0].std()
m_cran   = cranial_excess[cranial_excess > 0].mean()
s_cran   = cranial_excess[cranial_excess > 0].std()
m_total  = total_excess  [total_excess   > 0].mean()
s_total  = total_excess  [total_excess   > 0].std()

fmt_mm  = lambda v: f"{int(round(v))} mm" if pd.notna(v) else "-"
fmt_pct = lambda v: f"{int(round(v))}%"  if pd.notna(v) else "-"

summary = pd.DataFrame({
    "METRIC": [
        "n",
        "Mean_caudal_overscan_excess",
        "SD_caudal_overscan_excess",
        "Mean_cranial_overscan_excess",
        "SD_cranial_overscan_excess",
        "Mean_total_overscan_excess",
        "SD_total_overscan_excess",
        "%_caudal_overscan",
        "%_cranial_overscan",
        "%_overscanning",
    ],
    "VALUE": [
        len(df),
        fmt_mm(m_caudal),
        fmt_mm(s_caudal),
        fmt_mm(m_cran),
        fmt_mm(s_cran),
        fmt_mm(m_total),
        fmt_mm(s_total),
        fmt_pct((df["caudal_overscan?"]  == "yes").mean() * 100),
        fmt_pct((df["cranial_overscan?"] == "yes").mean() * 100),
        fmt_pct((df["overscanning?"]    == "yes").mean() * 100),
    ],
})

summary.to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Summary statistics written → {SUMMARY_CSV_PATH}")


Main CSV updated.
Summary statistics written → D:\Ryan\test_discont_abdo\summary_statistics.csv


In [32]:
# BLOCK 7 – Figure generation

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

def save_fig(fig, name: str, out_dir: Path = NIFTI_DIR, exts=("png",), dpi=300, close=True):
    out_dir.mkdir(parents=True, exist_ok=True)
    for ext in exts:
        fig.savefig(out_dir / f"{name}.{ext}", dpi=dpi, bbox_inches="tight")
    if close:
        plt.close(fig)

if "df" not in globals():
    df = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="utf-8-sig")
    df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)

cranial = pd.to_numeric(df.get("calc_cranial_overscan_mm", pd.Series(dtype=float)), errors="coerce").fillna(0).values
caudal  = pd.to_numeric(df.get("calc_caudal_overscan_mm",  pd.Series(dtype=float)), errors="coerce").fillna(0).values
x_idx   = np.arange(len(cranial))

def safe_ylim(ax, upper):
    if np.isfinite(upper) and upper > 0:
        ax.set_ylim(-upper * 1.05, upper * 1.05)

# Scatter
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(x_idx,  cranial, s=10, label="Cranial")
ax.scatter(x_idx, -caudal,  s=10, label="Caudal")
max_abs = np.nanmax([np.abs(cranial).max() if cranial.size else 0,
                     np.abs(caudal ).max() if caudal .size else 0])
safe_ylim(ax, max_abs)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Case index")
ax.set_ylabel("Overscan excess (mm)\n(+ cranial / − caudal)")
ax.set_title("Cranial vs Caudal Overscan Excess")
ax.legend(frameon=False)
plt.tight_layout()
save_fig(fig, "scatter_cranial_caudal")
print(f"Scatterplot saved to {NIFTI_DIR / 'scatter_cranial_caudal.png'}")

# Boxplot
box_cols   = ["calc_cranial_overscan_mm", "calc_caudal_overscan_mm", "calc_total_overscan_mm"]
box_labels = ["Cranial", "Caudal", "Total"]
data       = [pd.to_numeric(df.get(c, pd.Series(dtype=float)), errors="coerce").dropna().values for c in box_cols]

fig, ax = plt.subplots(figsize=(5, 7))
if any(len(d) for d in data):
    bp = ax.boxplot(
        data, vert=True, whis=1.5, showmeans=True, meanline=True,
        showcaps=True, showfliers=True, widths=0.6, patch_artist=True,
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="black", linestyle="--", linewidth=1),
        whiskerprops=dict(color="black", linestyle="--", linewidth=1),
        capprops=dict(color="black", linewidth=1),
        flierprops=dict(marker="o", markersize=4, markerfacecolor="none",
                        markeredgecolor="black", alpha=0.8),
    )
    for patch in bp["boxes"]:
        patch.set_facecolor("#1f77b4")
        patch.set_edgecolor("black")

ax.set_xticks(range(1, len(box_labels) + 1))
ax.set_xticklabels(box_labels)
ax.set_ylabel("Overscan excess (mm)")
ax.set_title("Overscan Excess – Box & Whisker")
plt.tight_layout()
save_fig(fig, "box_cranial_caudal_total")
print(f"Box-and-whisker plot saved to {NIFTI_DIR / 'box_cranial_caudal_total.png'}")

# Bar
fig, ax = plt.subplots(figsize=(6, 4.5))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ax.bar(x_idx,  cranial, width=0.8, color=colors[0], label="Cranial")
ax.bar(x_idx, -caudal,  width=0.8, color=colors[1], label="Caudal")
max_abs = np.nanmax([np.abs(cranial).max() if cranial.size else 0,
                     np.abs(caudal ).max() if caudal .size else 0])
safe_ylim(ax, max_abs)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Case index")
ax.set_ylabel("Overscan excess (mm)\n(+ cranial / − caudal)")
ax.set_title("Cranial vs Caudal Overscan Excess – Bar Plot")
ax.legend(frameon=False)
plt.tight_layout()
save_fig(fig, "bar_cranial_caudal")
print(f"Bar plot saved to {NIFTI_DIR / 'bar_cranial_caudal.png'}")


Scatterplot saved to D:\Ryan\test_discont_abdo\scatter_cranial_caudal.png
Box-and-whisker plot saved to D:\Ryan\test_discont_abdo\box_cranial_caudal_total.png
Bar plot saved to D:\Ryan\test_discont_abdo\bar_cranial_caudal.png
